# LoRA: low-rank updates to frozen weights

The base weight stays frozen while two small matrices `A` and `B` learn a correction. With rank `r`, the trainable update is `(alpha/r) B A`.

This notebook uses synthetic data and does **not** fine-tune a pretrained LLM.

In [ ]:
import torch
from torch import nn
from lora_lab import LoRALinear

torch.manual_seed(0)
base = nn.Linear(8, 6)
model = LoRALinear(base, rank=2, alpha=2)
x = torch.randn(4, 8)
print("Adapter trainable parameters:", model.adapter_parameter_count())
print("Base parameters require grad?", any(p.requires_grad for p in base.parameters()))

## Initial equivalence

Since `B` starts at zero, the adapter initially reproduces the frozen base output.

In [ ]:
assert torch.allclose(model(x), base(x))
print("Initial outputs are identical")

## Learning a synthetic low-rank correction

In [ ]:
from lora_lab.demo import run
initial, final = run(steps=120, seed=4)
assert final < initial
print("Fraction of initial MSE remaining:", round(final / initial, 4))

## Merge for inference

Merging is recommended only when no optimizer update is happening. Test numerical equivalence before and after.

In [ ]:
with torch.no_grad():
    model.B.add_(0.1)
before = model(x).detach().clone()
model.merge()
assert torch.allclose(before, model(x), atol=1e-6)
model.unmerge()
assert torch.allclose(before, model(x), atol=1e-6)
print("Merge / unmerge passed")